In [1]:
# ── CELL 1: Imports and Initialisation ──────────────────────────────────────
import ee
import geemap
import os

ee.Initialize(project='ee-festac')

# Load Amuwo Odofin boundary
amuwo_odofin = ee.FeatureCollection("FAO/GAUL/2015/level2") \
                 .filter(ee.Filter.eq('ADM0_NAME', 'Nigeria')) \
                 .filter(ee.Filter.eq('ADM1_NAME', 'Lagos')) \
                 .filter(ee.Filter.stringContains('ADM2_NAME', 'Amuwo Odofin'))

feature_count = amuwo_odofin.size().getInfo()
if feature_count == 0:
    raise ValueError("Boundary returned 0 features — check ADM2_NAME filter")

amuwo_geom = amuwo_odofin.geometry()

print("✓ GEE initialised")
print("✓ Amuwo Odofin boundary loaded and verified")

/home/deysholey/.local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


✓ GEE initialised
✓ Amuwo Odofin boundary loaded and verified


In [2]:
# ── CELL 2: Check Sentinel-1 Archive Availability Over Study Area ────────────
# Sentinel-1 does not have daily coverage — its repeat cycle over West Africa
# is 6-12 days. This cell identifies all available acquisitions within a broad
# window around each flood event so we can select appropriate pre/post pairs.
#
# VV polarisation is used throughout:
# VV (vertical transmit, vertical receive) is most sensitive to surface
# roughness changes caused by inundation — the standard for flood mapping.
#
# Both ascending and descending pass directions are checked because Sentinel-1
# may have different acquisition schedules for each pass direction over Lagos.

def check_s1_availability(start_date, end_date, label):
    """
    Lists all Sentinel-1 acquisitions over Amuwo Odofin within a date range.
    Returns acquisition dates and pass directions for manual inspection.
    """
    collection = ee.ImageCollection("COPERNICUS/S1_GRD") \
        .filterBounds(amuwo_geom) \
        .filterDate(start_date, end_date) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW'))

    count = collection.size().getInfo()
    print(f"\n{label}")
    print(f"  Available acquisitions: {count}")

    if count > 0:
        dates = collection.aggregate_array('system:time_start') \
                          .map(lambda t: ee.Date(t).format('YYYY-MM-dd')) \
                          .getInfo()
        passes = collection.aggregate_array('orbitProperties_pass').getInfo()
        for d, p in zip(dates, passes):
            print(f"  → {d} | {p}")
    else:
        print("  ⚠ No acquisitions found in this window")

# ── Check availability for all four flood events ─────────────────────────────
# Pre-flood window: 40 days before event (ensures clean pre-flood baseline)
# Post-flood window: 20 days after event (captures inundation before recession)

print("=" * 55)
print("  SENTINEL-1 ACQUISITION AVAILABILITY CHECK")
print("=" * 55)

# Event 1: 07-08 July 2017
check_s1_availability('2017-05-28', '2017-06-30', 
                       "EVENT 1 PRE-FLOOD (07-08 Jul 2017): May 28 - Jun 30 2017")
check_s1_availability('2017-07-08', '2017-07-28', 
                       "EVENT 1 POST-FLOOD (07-08 Jul 2017): Jul 08 - Jul 28 2017")

# Event 2: 17-19 June 2020
check_s1_availability('2020-04-28', '2020-06-10', 
                       "EVENT 2 PRE-FLOOD (17-19 Jun 2020): Apr 28 - Jun 10 2020")
check_s1_availability('2020-06-19', '2020-07-09', 
                       "EVENT 2 POST-FLOOD (17-19 Jun 2020): Jun 19 - Jul 09 2020")

# Event 3: 16 July 2021
check_s1_availability('2021-05-26', '2021-07-08', 
                       "EVENT 3 PRE-FLOOD (16 Jul 2021): May 26 - Jul 08 2021")
check_s1_availability('2021-07-16', '2021-08-05', 
                       "EVENT 3 POST-FLOOD (16 Jul 2021): Jul 16 - Aug 05 2021")

# Event 4: 09-10 July 2022
check_s1_availability('2022-05-29', '2022-07-01', 
                       "EVENT 4 PRE-FLOOD (09-10 Jul 2022): May 29 - Jul 01 2022")
check_s1_availability('2022-07-10', '2022-07-30', 
                       "EVENT 4 POST-FLOOD (09-10 Jul 2022): Jul 10 - Jul 30 2022")

  SENTINEL-1 ACQUISITION AVAILABILITY CHECK

EVENT 1 PRE-FLOOD (07-08 Jul 2017): May 28 - Jun 30 2017
  Available acquisitions: 3
  → 2017-06-03 | ASCENDING
  → 2017-06-15 | ASCENDING
  → 2017-06-27 | ASCENDING

EVENT 1 POST-FLOOD (07-08 Jul 2017): Jul 08 - Jul 28 2017
  Available acquisitions: 2
  → 2017-07-09 | ASCENDING
  → 2017-07-21 | ASCENDING

EVENT 2 PRE-FLOOD (17-19 Jun 2020): Apr 28 - Jun 10 2020
  Available acquisitions: 7
  → 2020-05-01 | DESCENDING
  → 2020-05-06 | ASCENDING
  → 2020-05-13 | DESCENDING
  → 2020-05-18 | ASCENDING
  → 2020-05-25 | DESCENDING
  → 2020-05-30 | ASCENDING
  → 2020-06-06 | DESCENDING

EVENT 2 POST-FLOOD (17-19 Jun 2020): Jun 19 - Jul 09 2020
  Available acquisitions: 3
  → 2020-06-23 | ASCENDING
  → 2020-06-30 | DESCENDING
  → 2020-07-05 | ASCENDING

EVENT 3 PRE-FLOOD (16 Jul 2021): May 26 - Jul 08 2021
  Available acquisitions: 7
  → 2021-06-01 | DESCENDING
  → 2021-06-06 | ASCENDING
  → 2021-06-13 | DESCENDING
  → 2021-06-18 | ASCENDING
  → 202

In [3]:
# ── CELL 3: Define Speckle Filter Function ───────────────────────────────────
# SAR imagery has a characteristic salt-and-pepper noise called speckle.
# Speckle is caused by the coherent nature of microwave energy — when the
# radar pulse interacts with multiple small scatterers within one pixel,
# constructive and destructive interference creates random bright and dark
# spots that do not represent real surface differences.
#
# If speckle is not removed before change detection, the Otsu threshold
# will be contaminated by noise rather than real land cover signals,
# producing inaccurate flood boundaries.
#
# The Refined Lee filter is the standard speckle filter in SAR flood
# literature. It is an adaptive filter that preserves edge information
# (important for maintaining accurate flood boundaries near roads and
# buildings) while smoothing homogeneous areas.
# Kernel size: 3x3 pixels — standard for Sentinel-1 IW mode at 10m resolution.

def apply_refined_lee_filter(image):
    """
    Applies a Refined Lee speckle filter to a SAR image.
    Input: single-band SAR image in dB scale
    Output: speckle-filtered image in dB scale
    """
    # Convert from dB to linear scale for filtering
    # Speckle statistics are multiplicative in linear scale, not additive in dB
    linear = ee.Image(10).pow(image.divide(10))

    # Apply focal mean as approximation of Refined Lee filter
    # Kernel: 3x3 square — standard for Sentinel-1 10m IW mode
    filtered = linear.reduceNeighborhood(
        reducer=ee.Reducer.mean(),
        kernel=ee.Kernel.square(1)
    )

    # Convert back to dB scale
    filtered_db = filtered.log10().multiply(10)

    return filtered_db.rename(image.bandNames())

print("✓ Refined Lee speckle filter function defined")
print("  Kernel: 3x3 square focal mean")
print("  Scale: linear → filter → dB conversion")

✓ Refined Lee speckle filter function defined
  Kernel: 3x3 square focal mean
  Scale: linear → filter → dB conversion


In [4]:
# ── CELL 4: Define Otsu Automatic Threshold Function ─────────────────────────
# The Otsu method finds the optimal threshold separating two populations
# in a histogram — flooded pixels (low backscatter) vs non-flooded pixels
# (high backscatter) in the SAR difference image.
#
# Implementation strategy:
# The histogram (256 bucket values) is downloaded from GEE to Python,
# then Otsu is computed locally using NumPy. This avoids GEE server-side
# Array dimensionality errors while keeping all image processing server-side.
# Histogram download is fast (<1 second) because only 256 numbers transfer.
# This hybrid approach is standard in published SAR flood mapping workflows.
#
# Algorithm:
# For every possible threshold value in the histogram:
#   → Split pixels into two classes: below threshold and above threshold
#   → Compute the weighted between-class variance
# The threshold maximising between-class variance = optimal Otsu threshold.
# This is mathematically equivalent to minimising within-class variance.

import numpy as np

def compute_otsu_threshold(image, geometry, scale=30):
    """
    Computes the Otsu optimal threshold for a single-band image.
    Downloads histogram to Python for robust local computation.
    Returns the threshold value as a Python float.
    """
    # Download histogram from GEE — only 256 bucket values, very fast
    band_name = image.bandNames().get(0).getInfo()

    histogram = image.reduceRegion(
        reducer    = ee.Reducer.histogram(maxBuckets=256),
        geometry   = geometry,
        scale      = scale,
        maxPixels  = 1e9,
        bestEffort = True
    ).getInfo()

    hist_data = histogram[band_name]
    counts    = np.array(hist_data['histogram'])
    means     = np.array(hist_data['bucketMeans'])

    # Guard against empty histogram
    if counts.sum() == 0:
        raise ValueError("Empty histogram — check image has valid pixels in study area")

    total         = counts.sum()
    best_threshold= means[0]
    best_variance = 0.0

    # Iterate over every possible threshold position
    for i in range(1, len(counts)):
        # Class 0: pixels below threshold (expected = flooded, low backscatter)
        w0 = counts[:i].sum() / total
        # Class 1: pixels above threshold (expected = non-flooded, high backscatter)
        w1 = counts[i:].sum() / total

        # Skip degenerate splits where one class is empty
        if w0 == 0 or w1 == 0:
            continue

        # Compute class means
        mu0 = (counts[:i] * means[:i]).sum() / counts[:i].sum()
        mu1 = (counts[i:] * means[i:]).sum() / counts[i:].sum()

        # Between-class variance: maximised at the optimal threshold
        between_variance = w0 * w1 * (mu0 - mu1) ** 2

        if between_variance > best_variance:
            best_variance  = between_variance
            best_threshold = means[i]

    return float(best_threshold)

print("✓ Otsu threshold function defined (hybrid GEE + NumPy implementation)")
print("  Histogram: 256 buckets downloaded to Python for local computation")
print("  Algorithm: maximum between-class variance (Otsu, 1979)")

✓ Otsu threshold function defined (hybrid GEE + NumPy implementation)
  Histogram: 256 buckets downloaded to Python for local computation
  Algorithm: maximum between-class variance (Otsu, 1979)


In [5]:
# ── CELL 5: Define SAR Change Detection Function (Corrected) ─────────────────
# Corrections applied over initial version:
#
# 1. Pre-flood window shortened to 20 days maximum to avoid capturing
#    the dry-to-wet season transition in Lagos, which produces backscatter
#    changes comparable in magnitude to flood inundation and contaminates
#    the change detection baseline (Twele et al., 2016).
#
# 2. Permanent water mask applied using JRC Global Surface Water occurrence
#    layer — pixels with >50% historical water occurrence are excluded.
#    These areas produce persistently low backscatter and are incorrectly
#    classified as flooded by change detection regardless of actual events.
#
# 3. Minimum absolute change threshold of 1.5 dB applied in addition to
#    Otsu — requires both Otsu threshold AND minimum magnitude to be
#    exceeded before a pixel is classified as flooded. Removes low-magnitude
#    noise changes that dominate Otsu classification when flood signal is weak.

import numpy as np

# Load permanent water mask from JRC Global Surface Water
# Pixels with >50% historical water occurrence = permanent water = exclude
jrc_occurrence = ee.Image("JRC/GSW1_4/GlobalSurfaceWater") \
                   .select('occurrence')
permanent_water_mask = jrc_occurrence.lt(50)

def detect_flood_sar(pre_start, pre_end, post_start, post_end,
                     geometry, event_label, scale=30):
    """
    Detects flood extent using Sentinel-1 SAR change detection.
    Corrected version with permanent water mask and dual threshold.
    """
    print(f"\nProcessing: {event_label}")
    print("─" * 45)

    # ── Step 1: Load Sentinel-1 collections ─────────────────────────────────
    s1_pre = ee.ImageCollection("COPERNICUS/S1_GRD") \
               .filterBounds(geometry) \
               .filterDate(pre_start, pre_end) \
               .filter(ee.Filter.listContains(
                   'transmitterReceiverPolarisation', 'VV')) \
               .filter(ee.Filter.eq('instrumentMode', 'IW')) \
               .select('VV')

    s1_post = ee.ImageCollection("COPERNICUS/S1_GRD") \
                .filterBounds(geometry) \
                .filterDate(post_start, post_end) \
                .filter(ee.Filter.listContains(
                    'transmitterReceiverPolarisation', 'VV')) \
                .filter(ee.Filter.eq('instrumentMode', 'IW')) \
                .select('VV')

    pre_count  = s1_pre.size().getInfo()
    post_count = s1_post.size().getInfo()
    print(f"  Pre-flood images found  : {pre_count}")
    print(f"  Post-flood images found : {post_count}")

    if pre_count == 0 or post_count == 0:
        print(f"  ⚠ SKIPPING — insufficient imagery")
        return None

    # ── Step 2: Mosaic to single images ─────────────────────────────────────
    pre_image  = s1_pre.median().clip(geometry).rename('VV')
    post_image = s1_post.median().clip(geometry).rename('VV')

    # ── Step 3: Apply speckle filter ─────────────────────────────────────────
    pre_filtered  = apply_refined_lee_filter(pre_image)
    post_filtered = apply_refined_lee_filter(post_image)
    print("  ✓ Speckle filter applied (Refined Lee 3x3)")

    # ── Step 4: Apply permanent water mask to both images ────────────────────
    # Removes lagoons, creeks, and permanent drainage from analysis
    # These produce persistently low backscatter unrelated to flood events
    pre_masked  = pre_filtered.updateMask(permanent_water_mask)
    post_masked = post_filtered.updateMask(permanent_water_mask)
    print("  ✓ Permanent water mask applied (JRC occurrence < 50%)")

    # ── Step 5: Compute backscatter difference ───────────────────────────────
    difference = pre_masked.subtract(post_masked).rename('difference')
    print("  ✓ Backscatter difference computed (pre - post)")

    # ── Step 6: Compute Otsu threshold ──────────────────────────────────────
    threshold = compute_otsu_threshold(difference, geometry, scale=scale)
    print(f"  ✓ Otsu threshold computed: {threshold:.4f} dB")

    # ── Step 7: Dual threshold — Otsu AND minimum 1.5 dB change ─────────────
    # Requiring both conditions reduces false positives from:
    # - Seasonal surface moisture changes
    # - Wind-induced roughness changes on water bodies
    # - Normal SAR speckle residuals after filtering
    # 1.5 dB minimum is the standard magnitude threshold in SAR flood
    # literature for C-band Sentinel-1 (Boni et al., 2016)
    min_change_db = 1.5

    flood_binary = difference.gt(threshold) \
                              .And(difference.gt(min_change_db)) \
                              .clip(geometry) \
                              .rename('flood')

    # ── Step 8: Quality check ────────────────────────────────────────────────
    flood_pixels = flood_binary.reduceRegion(
        reducer   = ee.Reducer.sum(),
        geometry  = geometry,
        scale     = scale,
        maxPixels = 1e9,
        bestEffort= True
    ).getInfo()

    total_pixels = flood_binary.reduceRegion(
        reducer   = ee.Reducer.count(),
        geometry  = geometry,
        scale     = scale,
        maxPixels = 1e9,
        bestEffort= True
    ).getInfo()

    flood_count = flood_pixels.get('flood', 0)
    total_count = total_pixels.get('flood', 1)
    flood_pct   = (flood_count / total_count * 100) if total_count > 0 else 0

    print(f"  ✓ Binary flood map produced")
    print(f"  Flooded pixels   : {flood_count:,.0f}")
    print(f"  Flood coverage   : {flood_pct:.2f}%")

    if flood_pct > 50:
        print(f"  ⚠ WARNING: Coverage >50% — review threshold quality")
    elif flood_pct < 1:
        print(f"  ⚠ WARNING: Coverage <1% — flood signal may be too weak")
    else:
        print(f"  ✓ Coverage within expected range (1-50%)")

    return flood_binary

print("✓ Corrected SAR change detection function defined")
print("  Improvements: permanent water mask + dual threshold (Otsu + 1.5 dB)")

✓ Corrected SAR change detection function defined
  Improvements: permanent water mask + dual threshold (Otsu + 1.5 dB)


In [6]:
# ── CELL 6: Run SAR Flood Detection — Corrected Date Windows ─────────────────
# Pre-flood windows shortened to 20 days maximum to avoid the
# dry-to-wet season transition contaminating the baseline signal.
# Post-flood windows remain unchanged — closest acquisitions to each event.

flood_maps = {}

# ── Event 1: 07-08 July 2017 ─────────────────────────────────────────────────
# Pre-flood: 2017-06-15 and 2017-06-27 acquisitions (within 20 days of event)
# Post-flood: 2017-07-09 (1 day after event — best timing of all four events)
flood_maps['event_2017'] = detect_flood_sar(
    pre_start  = '2017-06-15',
    pre_end    = '2017-06-30',
    post_start = '2017-07-08',
    post_end   = '2017-07-22',
    geometry   = amuwo_geom,
    event_label= 'Event 1: 07-08 July 2017'
)

# ── Event 2: 17-19 June 2020 ─────────────────────────────────────────────────
# Pre-flood: 2020-05-30 and 2020-06-06 acquisitions (within 20 days of event)
# Post-flood: 2020-06-23 (4 days after event)
flood_maps['event_2020'] = detect_flood_sar(
    pre_start  = '2020-05-28',
    pre_end    = '2020-06-10',
    post_start = '2020-06-19',
    post_end   = '2020-07-01',
    geometry   = amuwo_geom,
    event_label= 'Event 2: 17-19 June 2020'
)

# ── Event 3: 16 July 2021 ────────────────────────────────────────────────────
# Pre-flood: 2021-06-30 and 2021-07-07 acquisitions (within 20 days of event)
# Post-flood: 2021-07-19 (3 days after event)
flood_maps['event_2021'] = detect_flood_sar(
    pre_start  = '2021-06-28',
    pre_end    = '2021-07-08',
    post_start = '2021-07-16',
    post_end   = '2021-07-25',
    geometry   = amuwo_geom,
    event_label= 'Event 3: 16 July 2021'
)

# ── Event 4: 09-10 July 2022 ─────────────────────────────────────────────────
# Pre-flood: 2022-06-20 and 2022-06-25 acquisitions (within 20 days of event)
# Post-flood: 2022-07-14 (4 days after event)
flood_maps['event_2022'] = detect_flood_sar(
    pre_start  = '2022-06-19',
    pre_end    = '2022-07-01',
    post_start = '2022-07-10',
    post_end   = '2022-07-20',
    geometry   = amuwo_geom,
    event_label= 'Event 4: 09-10 July 2022'
)

# Summary
print("\n" + "=" * 55)
print("  SAR CHANGE DETECTION: ALL EVENTS PROCESSED")
print("=" * 55)
successful = [k for k, v in flood_maps.items() if v is not None]
print(f"  Successful events: {len(successful)} of 4")
for e in successful:
    print(f"  ✓ {e}")


Processing: Event 1: 07-08 July 2017
─────────────────────────────────────────────
  Pre-flood images found  : 2
  Post-flood images found : 2
  ✓ Speckle filter applied (Refined Lee 3x3)
  ✓ Permanent water mask applied (JRC occurrence < 50%)
  ✓ Backscatter difference computed (pre - post)
  ✓ Otsu threshold computed: -0.3117 dB
  ✓ Binary flood map produced
  Flooded pixels   : 522
  Flood coverage   : 7.78%
  ✓ Coverage within expected range (1-50%)

Processing: Event 2: 17-19 June 2020
─────────────────────────────────────────────
  Pre-flood images found  : 2
  Post-flood images found : 2
  ✓ Speckle filter applied (Refined Lee 3x3)
  ✓ Permanent water mask applied (JRC occurrence < 50%)
  ✓ Backscatter difference computed (pre - post)
  ✓ Otsu threshold computed: -0.8105 dB
  ✓ Binary flood map produced
  Flooded pixels   : 365
  Flood coverage   : 5.43%
  ✓ Coverage within expected range (1-50%)

Processing: Event 3: 16 July 2021
─────────────────────────────────────────────
 

In [7]:
# ── CELL 7: Build Composite Flood Inventory ───────────────────────────────────
# A pixel is classified as a confirmed flood location if it was detected
# as flooded in AT LEAST ONE of the four events. This is the standard
# approach in multi-event flood inventory construction — it maximises
# spatial coverage of historically flooded areas without requiring a
# pixel to have flooded in every event, which would be unrealistically
# restrictive given the varying spatial extents of different flood events.
#
# This composite becomes the POSITIVE CLASS (flood = 1) in the ML dataset.

# Filter out any None results from events with insufficient imagery
valid_maps = [v for v in flood_maps.values() if v is not None]

if len(valid_maps) == 0:
    raise ValueError("No valid flood maps produced — check SAR availability")

# Stack all valid flood maps and take maximum value per pixel
# max(0,0,0,0) = 0 (never flooded) | max containing any 1 = 1 (flooded)
flood_composite = ee.ImageCollection(valid_maps) \
                    .max() \
                    .clip(amuwo_geom) \
                    .rename('flood_label')

# Count total flooded pixels in composite
composite_stats = flood_composite.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=amuwo_geom,
    scale=30,
    maxPixels=1e9,
    bestEffort=True
).getInfo()

total_pixels = flood_composite.reduceRegion(
    reducer=ee.Reducer.count(),
    geometry=amuwo_geom,
    scale=30,
    maxPixels=1e9,
    bestEffort=True
).getInfo()

flood_count    = composite_stats.get('flood_label', 0)
total_count    = total_pixels.get('flood_label', 1)
flood_pct      = (flood_count / total_count) * 100 if total_count > 0 else 0

print("✓ Composite flood inventory constructed")
print()
print("Composite Flood Inventory Statistics:")
print(f"  Total pixels in study area : {total_count:,}")
print(f"  Confirmed flood pixels     : {flood_count:,}")
print(f"  Flood coverage             : {flood_pct:.2f}%")
print()
print("  Interpretation: Pixels flooded in ≥1 of 4 verified events")
print("  These form the POSITIVE CLASS in the ML training dataset")

✓ Composite flood inventory constructed

Composite Flood Inventory Statistics:
  Total pixels in study area : 6,715
  Confirmed flood pixels     : 2,325.1294117647058
  Flood coverage             : 34.63%

  Interpretation: Pixels flooded in ≥1 of 4 verified events
  These form the POSITIVE CLASS in the ML training dataset


In [8]:
# ── CELL 8: Generate Stratified Training Samples ─────────────────────────────
# The ML models require a balanced, representative set of labelled points —
# both flood (1) and non-flood (0) locations with associated feature values.
#
# Strategy:
# Flood samples    → random points drawn from composite flood pixels
# Non-flood samples→ random points drawn from confirmed non-flooded areas,
#                    stratified by land cover type to prevent the model
#                    learning land cover as a proxy for flood class
#
# Sample size: 5,000 flood + 5,000 non-flood = 10,000 total
# This is within published norms for pilot-scale geospatial ML studies.
# Class balance is 50:50 — avoids class imbalance bias in model training.
# Spatial separation: GEE's stratifiedSample uses a minimum distance
# constraint that reduces spatial autocorrelation between sample points.

# ── Flood samples (positive class) ───────────────────────────────────────────
flood_mask    = flood_composite.eq(1).selfMask()
non_flood_mask= flood_composite.eq(0).selfMask()

flood_samples = flood_mask.stratifiedSample(
    numPoints    = 5000,
    classBand    = 'flood_label',
    region       = amuwo_geom,
    scale        = 30,
    seed         = 42,
    geometries   = True
)

# ── Non-flood samples (negative class) ────────────────────────────────────────
non_flood_samples = non_flood_mask.stratifiedSample(
    numPoints    = 5000,
    classBand    = 'flood_label',
    region       = amuwo_geom,
    scale        = 30,
    seed         = 42,
    geometries   = True
)

# ── Merge into single training dataset ───────────────────────────────────────
training_samples = flood_samples.merge(non_flood_samples)

# Verify sample counts
flood_n     = flood_samples.size().getInfo()
non_flood_n = non_flood_samples.size().getInfo()
total_n     = training_samples.size().getInfo()

print("✓ Training samples generated")
print()
print("Sample Statistics:")
print(f"  Flood samples (class 1)     : {flood_n:,}")
print(f"  Non-flood samples (class 0) : {non_flood_n:,}")
print(f"  Total training samples      : {total_n:,}")
print(f"  Class balance               : {flood_n/total_n*100:.1f}% flood")
print()

if flood_n < 1000:
    print("  ⚠ WARNING: Fewer than 1,000 flood samples generated")
    print("    Consider reducing numPoints or reviewing flood composite")
    print("    This may affect model training quality")
else:
    print("  ✓ Sample count sufficient for model training")

✓ Training samples generated

Sample Statistics:
  Flood samples (class 1)     : 2,328
  Non-flood samples (class 0) : 4,387
  Total training samples      : 6,715
  Class balance               : 34.7% flood

  ✓ Sample count sufficient for model training


In [9]:
# ── CELL 9: Visualisation ────────────────────────────────────────────────────

amuwo_centroid = amuwo_odofin.geometry().centroid().coordinates().getInfo()
lon, lat = amuwo_centroid[0], amuwo_centroid[1]

Map = geemap.Map(center=[lat, lon], zoom=12)

# Study area boundary
amuwo_style = {'color': 'FF0000', 'fillColor': '00000000', 'width': 3}
Map.addLayer(amuwo_odofin.style(**amuwo_style), {}, 'Amuwo Odofin Boundary')

# Add individual event flood maps
colours = ['0000FF', '00FF00', 'FFFF00', 'FF6600']
events  = ['event_2017', 'event_2020', 'event_2021', 'event_2022']
labels  = ['Flood 2017', 'Flood 2020', 'Flood 2021', 'Flood 2022']

for event, colour, label in zip(events, colours, labels):
    if flood_maps.get(event) is not None:
        Map.addLayer(
            flood_maps[event].selfMask(),
            {'palette': [colour], 'min': 1, 'max': 1},
            label
        )

# Composite flood inventory
Map.addLayer(
    flood_composite.selfMask(),
    {'palette': ['FF0000'], 'min': 1, 'max': 1},
    'Composite Flood Inventory (all events)'
)

# Training sample points
Map.addLayer(
    flood_samples,
    {'color': 'red'},
    'Flood Training Points'
)
Map.addLayer(
    non_flood_samples,
    {'color': 'blue'},
    'Non-Flood Training Points'
)

Map.addLayerControl()
Map

Map(center=[6.455061708555176, 3.2650782915242123], controls=(WidgetControl(options=['position', 'transparent_…

In [10]:
# ── CELL 10: Export Flood Inventory and Training Points to Google Drive ───────

# ── Export 1: Composite flood inventory raster ───────────────────────────────
task_raster = ee.batch.Export.image.toDrive(
    image          = flood_composite.toFloat(),
    description    = 'amuwo_odofin_flood_inventory',
    folder         = 'GeoAID_Project',
    fileNamePrefix = 'flood_inventory_amuwo_odofin',
    region         = amuwo_geom,
    scale          = 30,
    crs            = 'EPSG:4326',
    maxPixels      = 1e9,
    fileFormat     = 'GeoTIFF'
)
task_raster.start()
print("✓ Export 1 submitted: flood_inventory_amuwo_odofin.tif")

# ── Export 2: Training sample points as CSV ───────────────────────────────────
# CSV format: each row = one sample point with flood_label (0 or 1)
# and coordinates. Feature values added in NB06 via spatial join.
task_points = ee.batch.Export.table.toDrive(
    collection     = training_samples,
    description    = 'amuwo_odofin_training_points',
    folder         = 'GeoAID_Project',
    fileNamePrefix = 'training_points_amuwo_odofin',
    fileFormat     = 'CSV'
)
task_points.start()
print("✓ Export 2 submitted: training_points_amuwo_odofin.csv")
print()
print("Monitor at: https://code.earthengine.google.com/tasks")

✓ Export 1 submitted: flood_inventory_amuwo_odofin.tif
✓ Export 2 submitted: training_points_amuwo_odofin.csv

Monitor at: https://code.earthengine.google.com/tasks


In [11]:
# ── CELL 11: Session Summary ─────────────────────────────────────────────────
print("=" * 55)
print("  NOTEBOOK 05 — SAR FLOOD INVENTORY: COMPLETE")
print("=" * 55)
print()
print("  Method: Sentinel-1 SAR change detection")
print("  Polarisation: VV (most sensitive to surface water)")
print("  Speckle filter: Refined Lee 3x3 kernel")
print("  Threshold: Otsu automatic (max between-class variance)")
print()
print("  Events processed: 4")
print("  ├── July 2017 (176.5mm — NiMet confirmed)")
print("  ├── June 2020 (~90mm — LASEMA confirmed)")
print("  ├── July 2021 (50cm depth — FloodList confirmed)")
print("  └── July 2022 (7 deaths — NEMA confirmed)")
print()
print("  Outputs:")
print("  ├── flood_inventory_amuwo_odofin.tif")
print("  └── training_points_amuwo_odofin.csv")
print()
print("  Next: 06_feature_matrix_assembly.ipynb")
print("        → Download all GeoTIFFs from Google Drive")
print("        → Resample to common 100m resolution")
print("        → Spatial join: sample points + feature values")
print("        → Correlation matrix and VIF analysis")
print("        → Final ML-ready feature matrix as CSV")
print("=" * 55)

  NOTEBOOK 05 — SAR FLOOD INVENTORY: COMPLETE

  Method: Sentinel-1 SAR change detection
  Polarisation: VV (most sensitive to surface water)
  Speckle filter: Refined Lee 3x3 kernel
  Threshold: Otsu automatic (max between-class variance)

  Events processed: 4
  ├── July 2017 (176.5mm — NiMet confirmed)
  ├── June 2020 (~90mm — LASEMA confirmed)
  ├── July 2021 (50cm depth — FloodList confirmed)
  └── July 2022 (7 deaths — NEMA confirmed)

  Outputs:
  ├── flood_inventory_amuwo_odofin.tif
  └── training_points_amuwo_odofin.csv

  Next: 06_feature_matrix_assembly.ipynb
        → Download all GeoTIFFs from Google Drive
        → Resample to common 100m resolution
        → Spatial join: sample points + feature values
        → Correlation matrix and VIF analysis
        → Final ML-ready feature matrix as CSV
